# Ingestion — fiches d'identité des sources

Ce notebook analyse les sources de `datas/` :
- fichiers CSV (séparateur `,` `;` tabulation ou `|` détecté automatiquement) ;
- scripts SQL (`.sql`, `.sql.txt`) : schéma lu avec `sqlglot`, données rejouées dans un SQLite en mémoire via `SQLAlchemy`.

Il produit, pour chacun, une fiche d'identité Markdown dans
`artifacts/ingestion/identities/<AAAAMMJJ>/<nom_source>_<HHMMSS>.md`.

In [ ]:
import csv
from datetime import datetime
import re
from pathlib import Path

import pandas as pd
import sqlglot
from sqlalchemy import create_engine
from sqlglot import exp

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "datas"
OUTPUT_ROOT = PROJECT_DIR / "artifacts" / "ingestion" / "identities"

SOURCES = [
    DATA_DIR / "releves_incidents.csv",
    DATA_DIR / "telemetry.csv",
    DATA_DIR / "machine.sql.txt",
]

## Fonctions utilitaires

In [ ]:
def human_filesize(n_bytes: int) -> str:
    """Taille lisible, base 1024 : 9999.99 [o | Ko | Mo | Go | To]."""
    size = float(n_bytes)
    for unit in ["o", "Ko", "Mo", "Go"]:
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024
    return f"{size:.2f} To"


def detect_charset(path: Path) -> str:
    """Détection simple de l'encodage (sans dépendance externe)."""
    raw = path.read_bytes()
    if raw.startswith(b"\xef\xbb\xbf"):
        return "UTF8 (BOM)"
    if raw.startswith((b"\xff\xfe", b"\xfe\xff")):
        return "UTF16"
    try:
        raw.decode("ascii")
        return "ASCII"
    except UnicodeDecodeError:
        pass
    try:
        raw.decode("utf-8")
        return "UTF8"
    except UnicodeDecodeError:
        pass
    try:
        raw.decode("cp1252")
        return "CP1252"
    except UnicodeDecodeError:
        return "ISO8859-1"


PYTHON_ENCODING = {"UTF8 (BOM)": "utf-8-sig", "UTF16": "utf-16", "ASCII": "utf-8",
                   "UTF8": "utf-8", "CP1252": "cp1252", "ISO8859-1": "latin-1"}


CANDIDATE_SEPARATORS = [",", ";", "\t", "|"]
SEPARATOR_LABELS = {",": "virgule (,)", ";": "point-virgule (;)", "\t": "tabulation (\\t)", "|": "pipe (|)"}


def detect_separator(path: Path, encoding: str, sample_lines: int = 50) -> str:
    """Détecte le séparateur parmi , ; \t | .

    1. csv.Sniffer sur un échantillon (gère les guillemets) ;
    2. sinon, on retient le candidat présent le plus souvent avec un nombre
       constant d'occurrences sur chaque ligne de l'échantillon.
    """
    with path.open(encoding=encoding, newline="") as fh:
        lines = [line for _, line in zip(range(sample_lines), fh) if line.strip()]
    sample = "".join(lines)
    try:
        return csv.Sniffer().sniff(sample, delimiters="".join(CANDIDATE_SEPARATORS)).delimiter
    except csv.Error:
        pass
    best, best_score = ",", 0
    for sep in CANDIDATE_SEPARATORS:
        counts = [line.count(sep) for line in lines]
        if counts and counts[0] > 0 and len(set(counts)) == 1 and counts[0] > best_score:
            best, best_score = sep, counts[0]
    return best


DECIMAL_COMMA = re.compile(r"^-?\d+,\d+$")


def fix_decimal_comma(df: pd.DataFrame) -> pd.DataFrame:
    """Convertit en float les colonnes texte au format décimal français ("46,348")."""
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            continue
        values = df[col].dropna().astype(str).str.strip()
        if len(values) and values.str.match(r"^-?\d+([.,]\d+)?$").all() and values.str.match(DECIMAL_COMMA).any():
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    return df

In [ ]:
ID_PATTERN = re.compile(r"(^id$|_id$|^id_|_badge$|^badge)", re.IGNORECASE)
TIME_PATTERN = re.compile(r"^\d{1,2}:\d{2}(:\d{2})?$")
DATE_PATTERN = re.compile(r"^\d{4}-\d{2}-\d{2}$")
DATETIME_PATTERN = re.compile(r"^\d{4}-\d{2}-\d{2}[ T]\d{1,2}:\d{2}(:\d{2})?")

NUMERIC_TYPES = {"INTEGER", "FLOAT"}
TEMPORAL_TYPES = {"DATE", "DATETIME", "TIME"}


def _all_match(values: pd.Series, pattern: re.Pattern) -> bool:
    return len(values) > 0 and values.astype(str).str.match(pattern).all()


def infer_type(name: str, s: pd.Series) -> str:
    """Types : IDENTITY, BOOLEAN, INTEGER, FLOAT, DATE, DATETIME, TIME, CATEGORY, TEXT, EMPTY."""
    values = s.dropna()
    if values.empty:
        return "EMPTY"
    if ID_PATTERN.search(name):
        return "IDENTITY"
    if pd.api.types.is_bool_dtype(s) or (
        pd.api.types.is_numeric_dtype(s) and set(values.unique()) <= {0, 1}
    ):
        return "BOOLEAN"
    if pd.api.types.is_integer_dtype(s):
        return "INTEGER"
    if pd.api.types.is_float_dtype(s):
        return "INTEGER" if (values % 1 == 0).all() else "FLOAT"
    # colonnes texte : tentative de reconnaissance temporelle
    if _all_match(values, DATE_PATTERN):
        return "DATE"
    if _all_match(values, DATETIME_PATTERN):
        return "DATETIME"
    if _all_match(values, TIME_PATTERN):
        return "TIME"
    if values.nunique() <= max(20, 0.05 * len(values)):
        return "CATEGORY"
    return "TEXT"


def to_temporal(s: pd.Series, col_type: str) -> pd.Series:
    if col_type == "TIME":
        # "05:42" -> timedelta, pour un tri chronologique correct
        return pd.to_timedelta(s.astype(str).where(s.astype(str).str.count(":") == 2, s.astype(str) + ":00"))
    if col_type == "DATETIME":
        # format mixte + utc : gère les horodatages avec fuseau ("2025-06-15 10:00:00+00")
        return pd.to_datetime(s, errors="coerce", format="mixed", utc=True)
    return pd.to_datetime(s, errors="coerce")


def fmt_value(v, col_type: str) -> str:
    if pd.isna(v):
        return "N/A"
    if col_type == "DATE":
        return v.strftime("%Y-%m-%d")
    if col_type == "DATETIME":
        return v.strftime("%Y-%m-%d %H:%M:%S")
    if col_type == "TIME":
        total = int(v.total_seconds())
        return f"{total // 3600:02d}:{total % 3600 // 60:02d}:{total % 60:02d}"
    if col_type == "INTEGER" and float(v).is_integer():
        return f"{int(v)}"
    return f"{float(v):.3f}"

## Calcul des statistiques et rendu Markdown (communs CSV / SQL)

In [ ]:
def compute_stats(df: pd.DataFrame, col_types: dict[str, str], skip: set[str] = frozenset()) -> list[dict]:
    """Min / max pour les colonnes numériques et dates, moyenne / médiane pour les numériques."""
    stats = []
    for col, col_type in col_types.items():
        if col in skip:
            continue
        if col_type in NUMERIC_TYPES:
            s = pd.to_numeric(df[col], errors="coerce")
            stats.append({"col": col, "type": col_type,
                          "min": fmt_value(s.min(), col_type), "max": fmt_value(s.max(), col_type),
                          "mean": fmt_value(s.mean(), "FLOAT"), "median": fmt_value(s.median(), "FLOAT")})
        elif col_type in TEMPORAL_TYPES:
            s = to_temporal(df[col], col_type)
            stats.append({"col": col, "type": col_type,
                          "min": fmt_value(s.min(), col_type), "max": fmt_value(s.max(), col_type),
                          "mean": None, "median": None})
    return stats


def render_stats(stats: list[dict], level: str = "###") -> list[str]:
    lines = []
    for st in stats:
        lines += [f"{level} {st['col']} ({st['type']})", "",
                  f"- Col name: {st['col']}",
                  f"- Min value: {st['min']}",
                  f"- Max value: {st['max']}"]
        if st["mean"] is not None:
            lines += [f"- Mean value: {st['mean']}",
                      f"- Median value: {st['median']}"]
        lines.append("")
    if not stats:
        lines += ["_Aucune colonne numérique ou date._", ""]
    return lines


def file_header(path: Path, begin: datetime, end: datetime, charset: str) -> list[str]:
    return [
        f"# Identity — {path.name}",
        "",
        f"- Source: {path.resolve()}",
        f"- Filesize: {human_filesize(path.stat().st_size)}",
        f"- ProcessBeginAt: {begin:%H:%M:%S}",
        f"- ProcessEndAt: {end:%H:%M:%S}",
        f"- Charset: {charset}",
    ]


def write_identity(path: Path, begin: datetime, lines: list[str], output_root: Path) -> Path:
    out_dir = output_root / f"{begin:%Y%m%d}"
    out_dir.mkdir(parents=True, exist_ok=True)
    base_name = path.name.split(".")[0]          # machine.sql.txt -> machine
    out_path = out_dir / f"{base_name}_{begin:%H%M%S}.md"
    out_path.write_text("\n".join(lines), encoding="utf-8")
    return out_path

## Fiche d'identité d'un CSV

In [ ]:
def build_csv_identity(path: Path, output_root: Path = OUTPUT_ROOT) -> Path:
    begin = datetime.now()

    charset = detect_charset(path)
    encoding = PYTHON_ENCODING[charset]
    separator = detect_separator(path, encoding)
    df = pd.read_csv(path, sep=separator, encoding=encoding)
    df = fix_decimal_comma(df)

    col_types = {col: infer_type(col, df[col]) for col in df.columns}
    stats = compute_stats(df, col_types)

    end = datetime.now()

    lines = file_header(path, begin, end, charset) + [
        f"- Separator: {SEPARATOR_LABELS.get(separator, repr(separator))}",
        f"- Rows Number: {len(df)}",
        f"- Cols Number: {df.shape[1]}",
        "",
        "## Column list",
        "",
        "| Colname | Infered Type |",
        "| ------- | ------------ |",
        *[f"| {col} | {t} |" for col, t in col_types.items()],
        "",
        "## Numeric & date columns",
        "",
        *render_stats(stats),
    ]
    return write_identity(path, begin, lines, output_root)

## Fiche d'identité d'un script SQL

1. `sqlglot` découpe le script (dialecte PostgreSQL) et fournit le schéma déclaré : types, clés, contraintes, index.
2. Les `CREATE TABLE` et `INSERT` sont traduits en SQLite et rejoués dans une base **en mémoire** via `SQLAlchemy`.
3. Chaque table est relue avec `pandas.read_sql_table` puis profilée comme un CSV.

Les colonnes absentes des `INSERT` (remplies par leur `DEFAULT`, ex. `created_at DEFAULT NOW()`) ne viennent pas de la source :
elles sont signalées et exclues des statistiques.

In [ ]:
SQL_TYPE_MAP = {
    "INTEGER": "INTEGER", "BIGINT": "INTEGER", "SMALLINT": "INTEGER", "TINYINT": "INTEGER", "SERIAL": "INTEGER", "BIGSERIAL": "INTEGER",
    "DECIMAL": "FLOAT", "NUMERIC": "FLOAT", "FLOAT": "FLOAT", "DOUBLE": "FLOAT", "REAL": "FLOAT", "MONEY": "FLOAT",
    "DATE": "DATE", "DATETIME": "DATETIME", "TIMESTAMP": "DATETIME", "TIMESTAMPTZ": "DATETIME", "TIMESTAMPLTZ": "DATETIME",
    "TIME": "TIME", "TIMETZ": "TIME", "BOOLEAN": "BOOLEAN",
}


def parse_sql_schema(statements: list[exp.Expression]) -> dict[str, dict]:
    """Schéma déclaré par table : colonnes (type SQL, contraintes), index, colonnes alimentées par INSERT."""
    schema: dict[str, dict] = {}
    for stmt in statements:
        if isinstance(stmt, exp.Create) and stmt.kind == "TABLE":
            table = stmt.this.this.name
            cols, pk = {}, []
            for item in stmt.this.expressions:
                if isinstance(item, exp.ColumnDef):
                    constraints = [c.kind.sql("postgres") for c in item.constraints]
                    is_pk = any(isinstance(c.kind, exp.PrimaryKeyColumnConstraint) for c in item.constraints)
                    is_fk = any(isinstance(c.kind, exp.Reference) for c in item.constraints)
                    cols[item.name] = {"sql_type": item.args["kind"].sql("postgres"),
                                       "sql_base": item.args["kind"].this.name,
                                       "constraints": constraints, "is_key": is_pk or is_fk}
                    if is_pk:
                        pk.append(item.name)
                elif isinstance(item, exp.PrimaryKey):
                    pk += [e.name for e in item.expressions]
            for name in pk:
                cols[name]["is_key"] = True
            schema[table] = {"columns": cols, "primary_key": pk, "indexes": [], "inserted": set()}
        elif isinstance(stmt, exp.Create) and stmt.kind == "INDEX":
            table = stmt.find(exp.Table).name
            if table in schema:
                cols = ", ".join(c.name for c in stmt.find_all(exp.Column))
                schema[table]["indexes"].append(f"{stmt.this.name} ({cols})")
        elif isinstance(stmt, exp.Insert):
            target = stmt.this
            table = target.this.name if isinstance(target, exp.Schema) else target.name
            if table in schema:
                inserted = [c.name for c in target.expressions] if isinstance(target, exp.Schema) else list(schema[table]["columns"])
                schema[table]["inserted"].update(inserted)
    return schema


def load_sql_in_memory(statements: list[exp.Expression]):
    """Rejoue CREATE TABLE / INSERT dans une base SQLite en mémoire (les index, inutiles au profilage, sont ignorés)."""
    engine = create_engine("sqlite://")
    with engine.begin() as conn:
        for stmt in statements:
            if isinstance(stmt, (exp.Transaction, exp.Commit, exp.Rollback)):
                continue
            if isinstance(stmt, exp.Create) and stmt.kind != "TABLE":
                continue
            # sqlglot ajoute "NULLS LAST" sur les colonnes de ON CONFLICT(...), refusé par SQLite
            stmt = stmt.transform(lambda node: node.this if isinstance(node, exp.Ordered) else node)
            conn.exec_driver_sql(stmt.sql(dialect="sqlite"))
    return engine


def infer_sql_type(name: str, meta: dict, s: pd.Series) -> str:
    if meta["is_key"] or ID_PATTERN.search(name):
        return "IDENTITY"
    mapped = SQL_TYPE_MAP.get(meta["sql_base"].upper())
    return mapped or infer_type(name, s)


def build_sql_identity(path: Path, output_root: Path = OUTPUT_ROOT, dialect: str = "postgres") -> Path:
    begin = datetime.now()

    charset = detect_charset(path)
    script = path.read_text(encoding=PYTHON_ENCODING[charset])
    statements = [s for s in sqlglot.parse(script, read=dialect) if s is not None]
    schema = parse_sql_schema(statements)
    engine = load_sql_in_memory(statements)

    sections, summary = [], []
    for table, info in schema.items():
        df = pd.read_sql_table(table, engine)
        col_types = {col: infer_sql_type(col, info["columns"][col], df[col]) for col in df.columns}
        not_from_source = {c for c in df.columns if c not in info["inserted"]}
        stats = compute_stats(df, col_types, skip=not_from_source)
        summary.append(f"| {table} | {len(df)} | {df.shape[1]} |")

        rows = []
        for col, t in col_types.items():
            meta = info["columns"][col]
            constraints = ", ".join(meta["constraints"]).replace("|", "\\|") or "-"
            origin = "DEFAULT (absente des INSERT)" if col in not_from_source else "INSERT"
            rows.append(f"| {col} | {meta['sql_type']} | {t} | {constraints} | {origin} |")

        sections += [
            f"## Table `{table}`",
            "",
            f"- Rows Number: {len(df)}",
            f"- Cols Number: {df.shape[1]}",
            f"- Primary key: {', '.join(info['primary_key']) or '-'}",
            f"- Indexes: {'; '.join(info['indexes']) or '-'}",
            "",
            "### Column list",
            "",
            "| Colname | SQL Type | Infered Type | Constraints | Origin |",
            "| ------- | -------- | ------------ | ----------- | ------ |",
            *rows,
            "",
            "### Numeric & date columns",
            "",
            *render_stats(stats, level="####"),
        ]
    engine.dispose()

    end = datetime.now()

    lines = file_header(path, begin, end, charset) + [
        f"- Format: SQL ({dialect})",
        f"- Statements Number: {len(statements)}",
        f"- Tables Number: {len(schema)}",
        f"- Rows Number: {sum(int(r.split('|')[2]) for r in summary)}",
        "",
        "| Table | Rows Number | Cols Number |",
        "| ----- | ----------- | ----------- |",
        *summary,
        "",
        *sections,
    ]
    return write_identity(path, begin, lines, output_root)

## Exécution

In [ ]:
def build_identity(path: Path, output_root: Path = OUTPUT_ROOT) -> Path:
    if ".sql" in path.suffixes:                  # machine.sql, machine.sql.txt
        return build_sql_identity(path, output_root)
    return build_csv_identity(path, output_root)


generated = [build_identity(src) for src in SOURCES]
for p in generated:
    print(p.relative_to(PROJECT_DIR))

In [ ]:
from IPython.display import Markdown, display

for p in generated:
    display(Markdown(p.read_text(encoding="utf-8")))